# Exploratory Data Analysis — Crypto vs. Traditional Assets (2014–2024)

This notebook performs EDA on the normalized performance data from the crypto-comparator project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Inspect the Dataset

In [ ]:
# Raw data extracted from crypto-comparator/src/app/page.tsx performanceData array
data = {
    'year': [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'Bitcoin':      [100, 135.2, 304.4, 4451.6, 1176.4, 2262.6, 9119.8, 14563.5, 5203.5, 13293.4, 29737.4],
    'Ethereum':     [100, 100, 878.5, 8129.0, 1429.0, 1387.1, 7925.8, 39591.4, 12860.2, 24494.6, 46236.6],
    'Solana':       [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 100, 11289.1, 659.6, 6752.3, 12653.6],
    'S&P 500':      [100, 99.3, 108.7, 129.9, 121.8, 157.0, 182.5, 231.5, 186.5, 231.8, 291.6],
    'NASDAQ':       [100, 108.3, 114.7, 150.9, 149.4, 206.0, 303.8, 391.3, 258.0, 396.7, 493.8],
    'Russell 2000': [100, 94.1, 112.7, 127.5, 111.9, 138.5, 164.2, 186.0, 145.8, 167.8, 183.8],
    'Real Estate':  [100, 105.4, 111.4, 118.1, 123.5, 128.3, 139.8, 163.3, 177.7, 187.3, 196.4],
    'Gold':         [100, 88.4, 96.1, 108.2, 106.7, 127.0, 158.3, 150.0, 152.1, 172.2, 279.5],
    'Silver':       [100, 86.5, 100.1, 105.6, 96.9, 111.8, 165.3, 145.9, 150.0, 149.5, 186.1],
    'WTI Oil':      [100, 69.5, 100.8, 113.4, 85.3, 114.6, 91.1, 141.2, 150.7, 134.5, 131.8]
}

df = pd.DataFrame(data).set_index('year')
print(f'Shape: {df.shape} ({df.shape[0]} years x {df.shape[1]} assets)')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nTotal missing cells: {df.isnull().sum().sum()} / {df.size} ({df.isnull().sum().sum()/df.size*100:.1f}%)')
df

## 2. Descriptive Statistics

In [ ]:
desc = df.describe().round(1)
desc

In [ ]:
# Year-over-year returns (%)
returns = df.pct_change() * 100
returns = returns.dropna(how='all')  # drop first row (all NaN from pct_change)
print('Year-over-Year Returns (%):')
returns.round(1)

In [ ]:
# Summary statistics of annual returns
return_stats = pd.DataFrame({
    'Mean Return (%)': returns.mean(),
    'Std Dev (%)': returns.std(),
    'Min Return (%)': returns.min(),
    'Max Return (%)': returns.max(),
    'Cumulative (%)': (df.iloc[-1] - 100)
}).round(1)

# Add category
categories = {
    'Bitcoin': 'Crypto', 'Ethereum': 'Crypto', 'Solana': 'Crypto',
    'S&P 500': 'Traditional', 'NASDAQ': 'Traditional', 'Russell 2000': 'Traditional', 'Real Estate': 'Traditional',
    'Gold': 'Commodity', 'Silver': 'Commodity', 'WTI Oil': 'Commodity'
}
return_stats['Category'] = return_stats.index.map(categories)
return_stats.sort_values('Mean Return (%)', ascending=False)

In [ ]:
# Mean return by category
print('Mean Annual Return by Category:')
return_stats.groupby('Category')[['Mean Return (%)', 'Std Dev (%)']].mean().round(1)

## 3. Correlation Matrix

In [ ]:
# Correlation on annual returns (excluding Solana due to limited data)
corr_assets = [c for c in returns.columns if c != 'Solana']
corr = returns[corr_assets].corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, vmin=-1, vmax=1, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Correlation Matrix of Annual Returns (2014–2024)', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr)

## 4. Normalized Growth Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Crypto
for col in ['Bitcoin', 'Ethereum', 'Solana']:
    axes[0].plot(df.index, df[col], marker='o', label=col)
axes[0].set_title('Crypto Assets')
axes[0].set_ylabel('Normalized Value (base 100)')
axes[0].legend()
axes[0].set_yscale('log')

# Traditional
for col in ['S&P 500', 'NASDAQ', 'Russell 2000', 'Real Estate']:
    axes[1].plot(df.index, df[col], marker='o', label=col)
axes[1].set_title('Traditional Assets')
axes[1].legend()

# Commodities
for col in ['Gold', 'Silver', 'WTI Oil']:
    axes[2].plot(df.index, df[col], marker='o', label=col)
axes[2].set_title('Commodities')
axes[2].legend()

for ax in axes:
    ax.set_xlabel('Year')
    ax.grid(True, alpha=0.3)

plt.suptitle('Asset Performance by Category (2014–2024, base 100)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('growth_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Volatility vs. Return (Risk-Return Profile)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = {'Crypto': '#FF6B35', 'Traditional': '#2563EB', 'Commodity': '#FFD700'}

for asset in return_stats.index:
    cat = return_stats.loc[asset, 'Category']
    ax.scatter(return_stats.loc[asset, 'Std Dev (%)'], return_stats.loc[asset, 'Mean Return (%)'],
              c=colors[cat], s=120, edgecolors='black', linewidth=0.5, zorder=5)
    ax.annotate(asset, (return_stats.loc[asset, 'Std Dev (%)'], return_stats.loc[asset, 'Mean Return (%)']),
               textcoords='offset points', xytext=(8, 5), fontsize=9)

# Legend
for cat, color in colors.items():
    ax.scatter([], [], c=color, s=100, label=cat, edgecolors='black', linewidth=0.5)
ax.legend(fontsize=11)

ax.set_xlabel('Standard Deviation of Annual Returns (%)', fontsize=12)
ax.set_ylabel('Mean Annual Return (%)', fontsize=12)
ax.set_title('Risk vs. Return Profile (2014–2024)', fontsize=14)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('risk_return_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Best and Worst Years

In [ ]:
print('Best year per asset (YoY return %):')
for col in returns.columns:
    valid = returns[col].dropna()
    if len(valid) > 0:
        best_yr = valid.idxmax()
        print(f'  {col:15s}: {best_yr} ({valid[best_yr]:+.1f}%)')

print('\nWorst year per asset (YoY return %):')
for col in returns.columns:
    valid = returns[col].dropna()
    if len(valid) > 0:
        worst_yr = valid.idxmin()
        print(f'  {col:15s}: {worst_yr} ({valid[worst_yr]:+.1f}%)')

## 7. Key Findings Summary

The computed statistics above will be summarized in the README EDA section.